![cyber_photo](cyber_photo.jpg)

Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [6]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [7]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
val_df.head(2)


,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,381,1,101,4026532232,3,15,0
1,378,378,1,100,4026532231,3,15,0


In [8]:
# Start coding here
# Use as many cells as you need

#Seperating Features from labels
tr_features = train_df.drop(columns="sus_label")
tr_labels = train_df["sus_label"]
te_features = test_df.drop(columns="sus_label")
te_labels = test_df["sus_label"]
v_features = val_df.drop(columns="sus_label")
v_labels = val_df["sus_label"]

print("Feature shape (rows, columns):", tr_features.shape)
print("Starting in_features:", tr_features.shape[1])
print("Unique label values:", tr_labels.unique())

Feature shape (rows, columns): (763144, 7)
Starting in_features: 7
Unique label values: [1 0]


In [9]:
#scaling the data using sklearn standard scaler
scaler = StandardScaler()
scaled_training = scaler.fit_transform(tr_features)
scaled_test = scaler.transform(te_features)
scaled_val = scaler.transform(v_features)

# converting the arrays to tensor

X_train = torch.tensor(scaled_training, dtype=torch.float32)
X_test = torch.tensor(scaled_test, dtype=torch.float32)
X_val = torch.tensor(scaled_val, dtype=torch.float32)

y_train = torch.tensor(tr_labels.values, dtype=torch.float32).unsqueeze(1)
y_val   = torch.tensor(v_labels.values, dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(te_labels.values, dtype=torch.float32).unsqueeze(1)


#Define the Model
model = nn.Sequential(
    nn.Linear(7, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=0.0001)

#The loop

for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    prediction = model(X_train)
    loss = criterion(prediction, y_train)
    loss.backward()
    optimizer.step()

acc = Accuracy(task="Binary")
model.eval()
with torch.no_grad():
    train_accuracy = float(acc(model(X_train), y_train))
    acc.reset()
    val_accuracy = float(acc(model(X_val), y_val))
    acc.reset()
    test_accuracy = float(acc(model(X_test), y_test))
    acc.reset()
print(f"Validation Accuracy is: {val_accuracy}")
print(f"Test Accuracy is: {test_accuracy}")
print(f"Training Accuracy is: {train_accuracy}")

Validation Accuracy is: 0.9958405494689941
Test Accuracy is: 0.09265109896659851
Training Accuracy is: 0.9983371496200562
